# Importações por NCM 2024

Daniel da Cunha Costa - 2024006064

Isaac Reyes Alves de Abreu - 2025050342

Pedro Luiz Siva - 2024006129

In [1]:
import csv
import pandas as pd
import sqlite3

import requests
import seaborn as sns
import matplotlib.pyplot as plt

## Funções auxiliares

In [5]:
def fetch(query, conn, formated=True):
    """ 
        Executa uma query SQL em um banco de dados SQLite e retorna o resultado. 
        Converte o resultado em um DataFrame do Pandas se formated for True.
    """
    
    cur = conn.cursor()
    cur.execute(query)
    rs = cur.fetchall()

    columns = [description[0] for description in cur.description]
    return pd.DataFrame(rs, columns=columns) if formated else rs

def show_tables(conn):
    """ 
        Retorna os nomes das tabelas de um banco de dados SQLite. 
    """

    return [x[0] for x in fetch("SELECT name FROM sqlite_master WHERE type='table'", conn, formated=False)]

def shape(table, conn):
    """ 
        Retorna o numero de linhas e colunas de uma tabela SQLite.
    """

    nrows = fetch(f"SELECT COUNT(*) FROM {table}", conn, formated=False)[0][0]
    ncols = fetch(f"SELECT COUNT(*) FROM pragma_table_info('{table}')", conn, formated=False)[0][0]

    return (nrows, ncols)

def desc(table, conn):
    """ 
        Retorna os nomes das colunas de uma tabela SQLite.
    """
    cur = conn.cursor()
    cur.execute(f"PRAGMA table_info({table})")
    columns = [row[1] for row in cur.fetchall()]

    return columns

def info(table, conn):
    """ 
    Sumario resumido do esquema e perfil de dados de uma tabela SQLite. 
       """
    # table constraints (domain, null, default, pk)
    df1 = fetch(f'PRAGMA table_info("{table}")', conn)
    columns = desc(table, conn)
    
    # entries per column
    counts = ', '.join([f'COUNT(*) AS "{column}"' for column in columns])
    df2 = fetch(f'SELECT {counts} FROM "{table}"', conn).transpose()
    df2.columns = ['count']
    
    # non-null entries per column
    counts = ', '.join([f'COUNT("{column}") AS "{column}"' for column in columns])
    df3 = fetch(f'SELECT {counts} FROM "{table}"', conn).transpose()
    df3.columns = ['notnull count']

    # unique non-null entries per column
    counts = ', '.join([f'COUNT(DISTINCT "{column}") AS "{column}"' for column in columns])
    df4 = fetch(f'SELECT {counts} FROM "{table}"', conn).transpose()
    df4.columns = ['unique count']
    
    return df1.merge(df2, left_on='name', right_index=True) \
            .merge(df3, left_on='name', right_index=True) \
            .merge(df4, left_on='name', right_index=True)



# Dados

Source dataset: https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm/IMP_2024.csv